In [ ]:
# pip install tensorflow scikit-learn pandas numpy matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# =============================================================
# 1. LOAD DATA
# =============================================================
df = pd.read_csv("data/processed/eurusd_merged.csv", parse_dates=["Date"])
df = df.set_index("Date")
series = df["EURUSD"].values.reshape(-1, 1)

print(f"Series length: {len(series)}")

In [ ]:
# =============================================================
# 2. TRAIN/TEST SPLIT (90/10)
# =============================================================
n = len(series)
n_train = int(0.9 * n)
train_raw = series[:n_train]
test_raw  = series[n_train:]

# Scale
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_raw)
test_scaled  = scaler.transform(test_raw)

In [ ]:
# =============================================================
# 3. CREATE SEQUENCES
# =============================================================
def create_sequences(data, lag=30):
    X, y = [], []
    for i in range(lag, len(data)):
        X.append(data[i-lag:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

LAG = 30
X_train, y_train = create_sequences(train_scaled, LAG)
X_test,  y_test  = create_sequences(test_scaled,  LAG)

X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test  = X_test.reshape((X_test.shape[0],  X_test.shape[1],  1))

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")


In [ ]:
# =============================================================
# 4. BUILD MODELS
# =============================================================
def build_lstm(lag):
    model = Sequential([
        LSTM(50, return_sequences=True, input_shape=(lag, 1)),
        Dropout(0.2),
        LSTM(50),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

def build_gru(lag):
    model = Sequential([
        GRU(50, return_sequences=True, input_shape=(lag, 1)),
        Dropout(0.2),
        GRU(50),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse")
    return model


In [ ]:
# =============================================================
# 5. TRAIN
# =============================================================
early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

model_lstm = build_lstm(LAG)
history_lstm = model_lstm.fit(X_train, y_train,
                              epochs=100, batch_size=32,
                              validation_split=0.2,
                              callbacks=[early_stop], verbose=1)

model_gru = build_gru(LAG)
history_gru = model_gru.fit(X_train, y_train,
                            epochs=100, batch_size=32,
                            validation_split=0.2,
                            callbacks=[early_stop], verbose=1)


In [ ]:
# =============================================================
# 6. LOSS CURVES
# =============================================================
plt.figure(figsize=(10, 5))
plt.plot(history_lstm.history["loss"], label="LSTM Train")
plt.plot(history_lstm.history["val_loss"], label="LSTM Val")
plt.plot(history_gru.history["loss"], label="GRU Train")
plt.plot(history_gru.history["val_loss"], label="GRU Val")
plt.legend()
plt.title("Training Loss Comparison")
plt.savefig("figures/12_lstm_gru_loss.png", dpi=120)
plt.close()

In [ ]:
# =============================================================
# 7. PREDICT
# =============================================================
pred_lstm = model_lstm.predict(X_test)
pred_gru  = model_gru.predict(X_test)

# Inverse transform
pred_lstm_inv = scaler.inverse_transform(pred_lstm)
pred_gru_inv  = scaler.inverse_transform(pred_gru)
y_test_inv    = scaler.inverse_transform(y_test.reshape(-1, 1))


In [ ]:
# =============================================================
# 8. METRICS
# =============================================================
def metrics(actual, pred, name):
    mse  = mean_squared_error(actual, pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(actual, pred)
    mape = np.mean(np.abs((actual - pred) / actual)) * 100
    print(f"{name} -> MSE: {mse:.6f} | RMSE: {rmse:.6f} | MAE: {mae:.6f} | MAPE: {mape:.4f}%")
    return {"Model": name, "MSE": mse, "RMSE": rmse, "MAE": mae, "MAPE": mape}

results = [
    metrics(y_test_inv, pred_lstm_inv, "LSTM"),
    metrics(y_test_inv, pred_gru_inv,  "GRU")
]
pd.DataFrame(results).to_csv("figures/metrics_lstm_gru.csv", index=False)


In [ ]:

# =============================================================
# 9. PLOT
# =============================================================
plt.figure(figsize=(14, 6))
plt.plot(y_test_inv, label="Actual", color="black")
plt.plot(pred_lstm_inv, label="LSTM", color="blue", alpha=0.7)
plt.plot(pred_gru_inv,  label="GRU",  color="red",  alpha=0.7)
plt.legend()
plt.title("LSTM vs GRU - EUR/USD Forecast")
plt.savefig("figures/13_lstm_gru_forecast.png", dpi=120)
plt.close()